# refitgap in three steps

1. Table 1: two scorers with identical AUROC and average precision whose deficits split in
   opposite ways.
2. One paired refit from stored scores, then the five ASRS and MultiEURLEX comparisons of
   Figure 2.
3. Your own models, on a tiny synthetic set in the public CSV format.

About 15 seconds on a laptop. Run it from a clone (steps 2 and 3 read files there), after
`pip install refitgap` or `pip install -e .`.

In [1]:
import json, sys
from pathlib import Path
import refitgap as rg

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "paper" / "results").is_dir())
print("refitgap", rg.__version__)

refitgap 0.2.0


## 1. Table 1

Fourteen records, one label, a threshold of 0.5 carried from the reference model. Final A keeps
the ranking but its scores slid, so the carried rule lands one record late; final B keeps the
cut but its positives slid down the ranking. Exact fractions, naturally.

In [2]:
ex = rg.identification_example()
print(f"{'scorer':10s} {'AUROC':>6s} {'AP':>6s} {'F1 carried':>11s} {'F1 refitted':>12s}")
for name, s in ex["scorers"].items():
    print(f"{name:10s} {str(s['auroc']):>6s} {str(s['average_precision']):>6s} {str(s['carried_f1']):>11s} {str(s['best_f1']):>12s}")
for name, d in ex["decomposition"].items():
    print(f"{name}: deficit {d['deficit']} = attainable gap {d['attainable_gap']} + change in mismatch {d['mismatch_change']}")

scorer      AUROC     AP  F1 carried  F1 refitted
reference    6/11   5/18         1/2          1/2
final A      6/11   5/18        6/13          1/2
final B      6/11   5/18        6/13         6/13
final A: deficit 1/26 = attainable gap 0 + change in mismatch 1/26
final B: deficit 1/26 = attainable gap 1/26 + change in mismatch 0


Same AUROC (6/11), same average precision (5/18), same deficit (1/26)... opposite
components. Only scores, labels and a refit tell them apart.

## 2. One paired refit from stored scores

Replay against sequential training, BiGRU, MultiEURLEX, seed 0, period 2 (id 1). `compare()`
gives the carried level U, the refitted level V (200 split-half partitions) and the split of
the deficit; the stored analysis must agree to 1e-12.

In [3]:
from refitgap.runs import load_run

folder = ROOT / "paper" / "results" / "multieurlex_bigru"
rep = load_run(folder / "eurlex_C2_replay_lam1_s0_scores.npz", periods=[1])
seq = load_run(folder / "eurlex_C2_naive_lam1_s0_scores.npz", periods=[1])   # stored files call sequential training "naive"
r = rg.compare(rep["Ps"], seq["Ps"], rep["Ys"], rep["thresholds"], seed=0, splits=200,
               period_ids=[1], labels_final=seq["Ys"], rule_name="inherited")
print("carried  U: replay %.4f  sequential %.4f" % (r["carried"]["ref"], r["carried"]["final"]))
print("refitted V: replay %.4f  sequential %.4f" % (r["refitted"]["ref"], r["refitted"]["final"]))
c = r["components"]
print("deficit of sequential against replay %+.4f = attainable gap %+.4f + change in mismatch %+.4f"
      % (c["deficit"], c["attainable_gap"], c["mismatch_change"]))

stored = json.loads((ROOT / "paper/results/paired_refit/paired_replay_eurlex.json").read_text())["pairs"]["0"]["per_period"]["1"]
diff = max(abs(r["carried"]["ref"] - stored["U_rep"]), abs(r["carried"]["final"] - stored["U_seq"]),
           abs(r["refitted"]["ref"] - stored["V_rep"]), abs(r["refitted"]["final"] - stored["V_seq"]))
print(f"largest difference from the stored analysis: {diff:.1e}")
assert diff < 1e-12

carried  U: replay 0.7092  sequential 0.6546
refitted V: replay 0.7146  sequential 0.6720
deficit of sequential against replay +0.0546 = attainable gap +0.0426 + change in mismatch +0.0120
largest difference from the stored analysis: 0.0e+00


Average over periods 2 to 7, pair over seeds, and you have Figure 2's rows on ASRS and
MultiEURLEX. From stored per-pair levels that takes a second (drop `--from-levels` to redo the
BiGRU pairs from raw scores: a few minutes). The three PubMed rows are stored in
`paper/results/pubmed_paired_refit/`; their score files do not ship.

In [4]:
sys.path.insert(0, str(ROOT / "examples" / "figure2"))
import reproduce_figure2
assert reproduce_figure2.main(["--from-levels"]) == 0

replay against                                          inherited              both refitted                     change   input
DistilBERT ASRS, sequential            +0.0109 [+0.0089, +0.0129] -0.0004 [-0.0030, +0.0022] -0.0113 [-0.0138, -0.0088]   stored per-pair levels
DistilBERT MultiEURLEX, sequential     +0.0241 [+0.0219, +0.0263] +0.0223 [+0.0196, +0.0250] -0.0018 [-0.0037, +0.0001]   stored per-pair levels
DistilBERT MultiEURLEX, extra updates  +0.0217 [+0.0179, +0.0255] +0.0198 [+0.0157, +0.0240] -0.0019 [-0.0027, -0.0011]   stored per-pair levels
BiGRU ASRS, sequential                 +0.0086 [+0.0021, +0.0159] +0.0034 [+0.0019, +0.0049] -0.0052 [-0.0122, +0.0008]   stored per-pair levels
BiGRU MultiEURLEX, sequential          +0.0305 [+0.0272, +0.0335] +0.0243 [+0.0220, +0.0271] -0.0062 [-0.0095, -0.0031]   stored per-pair levels
largest difference from expected.json: 3.3e-16 (tolerance 1e-10): match


## 3. Your own models

`examples/your_models/data/` holds labels, scores per method and seed, and the carried rule,
all as CSV. Swap in your own files and point the script at the folder.

In [5]:
sys.path.insert(0, str(ROOT / "examples" / "your_models"))
import compare_your_models
_ = compare_your_models.main([str(ROOT / "examples" / "your_models" / "data")])

mine against baseline: 4 labels, periods [1, 2, 3], seeds [0, 1, 2, 3], 200 partitions
 seed  G_U carried  G_V refitted    change
    0      +0.2148       +0.0594   -0.1553
    1      +0.2710       +0.0719   -0.1991
    2      +0.1819       +0.0779   -0.1040
    3      +0.1697       +0.0501   -0.1196
G_U carried   +0.2093  95% t interval [+0.1373, +0.2814]
G_V refitted  +0.0648  95% t interval [+0.0450, +0.0847]
change        -0.1445  95% t interval [-0.2118, -0.0772]


Most of the lead at the carried rule here is mismatch: once both refit, it shrinks from about
0.21 to about 0.06. Report both, and nobody gets fooled!